[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/openai/openai_agents_basic.ipynb)


# <center>OpenAI 智能体模式：基础智能体</center>

本入门指南介绍如何使用 `openai-agents` 库构建支持工具调用的基础智能体。

这里创建的基础智能体可以通过 `web_search` 回答股票相关问题。

### 安装库

In [ ]:
# 安装 OpenAI 基础库
!pip install -q openai openai-agents

# 安装 OpenInference/OpenTelemetry 追踪的可选库
!pip install -q arize-phoenix-otel openinference-instrumentation-openai-agents openinference-instrumentation-openai

### 设置密钥
将 OpenAI API 密钥添加到环境变量 `OPENAI_API_KEY`。

从 [app.phoenix.arize.com](https://app.phoenix.arize.com) 的设置页面复制 Phoenix `API_KEY`。

In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

### 设置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="openai-agents",
    auto_instrument=True,
)

## 创建基础智能体

In [ ]:
from agents import Agent, Runner, WebSearchTool

agent = Agent(
    name="Finance Agent",
    instructions="你是能够回答股票问题的金融智能体。使用网络搜索获取最新背景信息，然后用一句简短精炼的话回答。",
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4.1-mini",
)

In [ ]:
from pprint import pprint

result = await Runner.run(agent, "what is the latest news on Apple?")

# 获取最终输出
print(result.final_output)

# 获取生成最终输出时记录的完整消息列表
pprint(result.to_input_list())